# DWT–DCT–SVD hybrid image watermarking

Reference: *DWT-DCT-SVD: A Hybrid Image Watermarking Algorithm with FPP Resistant Enhancement* (Circuits, Systems, and Signal Processing, 2025), Sections 3.2–3.3.

**Pipeline (simplified faithful to the paper’s text):**
1. **Transformed watermark** — embed signature into biometric: DWT(LL) → DCT → SVD; modify biometric singular values with signature singular values; inverse SVD, DCT, DWT.
2. **Cover embedding** — per color channel: DWT(LL) → DCT → SVD; add scaled singular values of the transformed watermark; inverse transforms.
3. **Extraction (semi-blind)** — needs original **cover** and **biometric** to subtract $S_{VC}$, $S_{VB}$ and invert scaling ($\beta$, $\alpha$), as in §3.3.

Replace sample paths below with your own images (paper uses 512×512 cover, 256×256 biometric; biometric should be larger than signature).

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import dctn, idctn
from scipy.linalg import svd
from skimage import color, io, util
from skimage.transform import resize
import pywt

WAVELET = "db2"
WAVELET_MODE = "symmetric"

: 

In [ ]:
def to_float01(img: np.ndarray) -> np.ndarray:
    img = util.img_as_float(img)
    return np.clip(img, 0.0, 1.0)


def dct2(x: np.ndarray) -> np.ndarray:
    return dctn(x, type=2, norm="ortho")


def idct2(x: np.ndarray) -> np.ndarray:
    return idctn(x, type=2, norm="ortho")


def dwt_ll(channel: np.ndarray) -> tuple[np.ndarray, tuple]:
    """Single-level 2D DWT; return LL and coeffs for perfect reconstruction."""
    coeffs = pywt.dwt2(channel, WAVELET, mode=WAVELET_MODE)
    (LL, (LH, HL, HH)) = coeffs
    return LL, coeffs


def idwt_reconstruct(LL_new: np.ndarray, coeffs: tuple) -> np.ndarray:
    """Inverse DWT using modified LL and the original LH/HL/HH from `coeffs`."""
    _, (LH, HL, HH) = coeffs
    return pywt.idwt2((LL_new, (LH, HL, HH)), WAVELET, mode=WAVELET_MODE)


def svd_compact(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return U, s, Vt with x ≈ U @ diag(s) @ Vt (full matrices)."""
    U, s, Vt = svd(x, full_matrices=False)
    return U, s, Vt


def merge_singular(s_host: np.ndarray, s_wm: np.ndarray, scale: float) -> np.ndarray:
    """Additive merge on the shared singular spectrum (paper: apply SV of wm to host)."""
    k = min(len(s_host), len(s_wm))
    out = s_host.copy()
    out[:k] = s_host[:k] + scale * s_wm[:k]
    return out


def reconstruct_from_svd(U: np.ndarray, s: np.ndarray, Vt: np.ndarray) -> np.ndarray:
    return (U * s) @ Vt

In [ ]:
def embed_signature_in_biometric(
    biometric: np.ndarray,
    signature: np.ndarray,
    alpha: float = 0.08,
) -> tuple[np.ndarray, dict]:
    """
    Build transformed watermark TW (§3.2 part 1).
    Grayscale biometric/signature in [0,1], same height/width or resized to biometric shape.
    """
    bio = to_float01(biometric)
    if bio.ndim == 3:
        bio = color.rgb2gray(bio)
    sig = to_float01(signature)
    if sig.ndim == 3:
        sig = color.rgb2gray(sig)

    LL_b, coeffs_b = dwt_ll(bio)
    F_b = dct2(LL_b)
    Ub, sb, Vtb = svd_compact(F_b)

    sig_spatial = resize(sig, bio.shape, anti_aliasing=True)
    LL_s, _ = dwt_ll(sig_spatial)
    F_s = dct2(LL_s)
    _, ss, _ = svd_compact(F_s)

    sb_new = merge_singular(sb, ss, alpha)
    F_b_new = reconstruct_from_svd(Ub, sb_new, Vtb)
    LL_new = idct2(F_b_new)
    TW = idwt_reconstruct(LL_new, coeffs_b)
    TW = np.clip(TW, 0.0, 1.0)

    aux = {"Ub": Ub, "Vtb": Vtb, "coeffs_b": coeffs_b, "LL_b_shape": LL_b.shape}
    return TW, aux


def embed_tw_in_cover(
    cover_rgb: np.ndarray,
    TW_gray: np.ndarray,
    beta: float = 0.05,
) -> tuple[np.ndarray, dict]:
    """
    Embed transformed watermark into color cover (§3.2 part 2), per channel.
    """
    cover = to_float01(cover_rgb)
    if cover.ndim == 2:
        cover = np.stack([cover, cover, cover], axis=-1)

    h, w, _ = cover.shape
    TW = resize(to_float01(TW_gray), (h, w), anti_aliasing=True)

    out = np.zeros_like(cover)
    aux_channels: list[dict] = []

    for c in range(3):
        ch = cover[:, :, c]
        LL, coeffs = dwt_ll(ch)
        F = dct2(LL)
        U, s, Vt = svd_compact(F)

        LL_tw, _ = dwt_ll(TW)
        F_tw = dct2(LL_tw)
        _, s_tw, _ = svd_compact(F_tw)

        s_new = merge_singular(s, s_tw, beta)
        F_new = reconstruct_from_svd(U, s_new, Vt)
        LL_new = idct2(F_new)
        wmarked_ch = idwt_reconstruct(LL_new, coeffs)
        out[:, :, c] = np.clip(wmarked_ch, 0.0, 1.0)

        aux_channels.append({"U": U, "Vt": Vt, "s_orig": s.copy(), "coeffs": coeffs})

    return out, {"channels": aux_channels}


def psnr(a: np.ndarray, b: np.ndarray, data_range: float = 1.0) -> float:
    a = to_float01(a)
    b = to_float01(b)
    mse = np.mean((a - b) ** 2)
    if mse == 0:
        return float("inf")
    return 10.0 * np.log10((data_range**2) / mse)


def ncc(a: np.ndarray, b: np.ndarray) -> float:
    a = to_float01(a).ravel() - np.mean(to_float01(a))
    b = to_float01(b).ravel() - np.mean(to_float01(b))
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

In [ ]:
def extract_tw_from_watermarked(
    watermarked_rgb: np.ndarray,
    cover_rgb: np.ndarray,
    beta: float,
) -> np.ndarray:
    """
    §3.3 step 1: estimate singular spectrum of TW from watermarked vs cover, then rebuild
    an image-shaped TW (same reconstruction basis as embed, per channel average).
    """
    wm = to_float01(watermarked_rgb)
    cov = to_float01(cover_rgb)
    if cov.ndim == 2:
        cov = np.stack([cov, cov, cov], axis=-1)
    if wm.ndim == 2:
        wm = np.stack([wm, wm, wm], axis=-1)

    h, w, _ = wm.shape
    acc = np.zeros((h, w), dtype=np.float64)

    for c in range(3):
        ch_wm = wm[:, :, c]
        ch_co = cov[:, :, c]

        LL_w, coeff_w = dwt_ll(ch_wm)
        LL_c, coeff_c = dwt_ll(ch_co)
        F_w = dct2(LL_w)
        F_c = dct2(LL_c)
        Uw, sw, Vtw = svd_compact(F_w)
        Uc, sc, Vtc = svd_compact(F_c)

        k = min(len(sw), len(sc))
        s_tw_hat = (sw[:k] - sc[:k]) / beta
        s_tw_full = np.zeros_like(sw)
        s_tw_full[:k] = s_tw_hat

        # Use watermarked subspace for stable inversion (semi-blind variant)
        F_tw_hat = reconstruct_from_svd(Uw, s_tw_full, Vtw)
        LL_tw_hat = idct2(F_tw_hat)
        tw_ch = idwt_reconstruct(LL_tw_hat, coeff_w)
        acc += tw_ch

    TW_hat = np.clip(acc / 3.0, 0.0, 1.0)
    return TW_hat


def extract_signature_from_tw(
    TW_hat: np.ndarray,
    biometric: np.ndarray,
    alpha: float,
) -> np.ndarray:
    """§3.3 step 2: recover signature from TW and original biometric."""
    bio = to_float01(biometric)
    if bio.ndim == 3:
        bio = color.rgb2gray(bio)

    LL_b, coeffs_b = dwt_ll(bio)
    F_b = dct2(LL_b)
    Ub, sb, Vtb = svd_compact(F_b)

    TW = resize(to_float01(TW_hat), bio.shape, anti_aliasing=True)
    LL_t, coeffs_t = dwt_ll(TW)
    F_t = dct2(LL_t)
    _, st, Vtt = svd_compact(F_t)

    k = min(len(sb), len(st))
    ss_hat = (st[:k] - sb[:k]) / alpha
    ss_full = np.zeros_like(sb)
    ss_full[:k] = ss_hat

    # Reconstruct signature block in DCT(LL) domain then invert
    F_sig_hat = reconstruct_from_svd(Ub, ss_full, Vtb)
    LL_sig = idct2(F_sig_hat)
    sig_hat = idwt_reconstruct(LL_sig, coeffs_b)
    return np.clip(sig_hat, 0.0, 1.0)

## Demo: synthetic images (no files required)

Swap in `skimage.io.imread(...)` for real cover / biometric / signature.

In [ ]:
rng = np.random.default_rng(0)
cover = rng.random((512, 512, 3))
biometric = rng.random((256, 256))
signature = rng.random((128, 128))

alpha, beta = 0.08, 0.05

TW, _ = embed_signature_in_biometric(biometric, signature, alpha=alpha)
watermarked, _ = embed_tw_in_cover(cover, TW, beta=beta)

print(f"PSNR cover vs watermarked: {psnr(cover, watermarked):.2f} dB")

TW_ext = extract_tw_from_watermarked(watermarked, cover, beta=beta)
sig_ext = extract_signature_from_tw(TW_ext, biometric, alpha=alpha)

print(f"NCC(TW, TW_ext): {ncc(TW, resize(TW_ext, TW.shape, anti_aliasing=True)):.4f}")
print(
    f"NCC(signature, sig_ext): {ncc(signature, resize(sig_ext, signature.shape, anti_aliasing=True)):.4f}"
)

fig, ax = plt.subplots(2, 3, figsize=(10, 6))
ax[0, 0].imshow(cover)
ax[0, 0].set_title("Cover")
ax[0, 1].imshow(watermarked)
ax[0, 1].set_title("Watermarked")
ax[0, 2].imshow(TW, cmap="gray")
ax[0, 2].set_title("TW")
ax[1, 0].imshow(biometric, cmap="gray")
ax[1, 0].set_title("Biometric")
ax[1, 1].imshow(signature, cmap="gray")
ax[1, 1].set_title("Signature")
ax[1, 2].imshow(sig_ext, cmap="gray")
ax[1, 2].set_title("Extracted signature")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

## Optional: load your own files

```python
cover = io.imread("path/to/cover.png")
biometric = io.imread("path/to/fingerprint.png")
signature = io.imread("path/to/signature.png")
# then re-run the same alpha/beta cells
```